# ⚙️ 03 - Özellik Mühendisliği (Feature Engineering)
## Uydu Telemetri Anomali Tespiti

**Amaç:** Ön işleme adımında (02) temizlenen ve ölçeklendirilen verilerden (zaman serileri) makine öğrenmesi modellerinin (özellikle anomali tespiti) daha iyi öğrenebilmesi için zenginleştirilmiş, fiziksel anlamı olan (Reaction Wheels) özellikler (features) çıkarmak.

### Adımlar:
1. Zaman Alanı Özellikleri (Time Domain)
2. Frekans Alanı Özellikleri (Frequency Domain)
3. Reaction Wheel Özel Fiziksel Özellikler
4. Çok Değişkenli Özellikler (Multivariate)
5. Gecikmeli Özellikler (Lag Features)
6. Özellik Seçimi (Feature Selection - Variance & Correlation)
7. Özellik Doğrulama (t-SNE / PCA ile Görselleştirme)
8. Özellik Matrisini Kaydetme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import json
import sys
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# Kendi modülümüzü import ediyoruz
sys.path.insert(0, '..')
from src.feature_engineer import ReactionWheelFeatureEngineer

print('✅ Kütüphaneler ve Feature Engineer yüklendi.')

---
## 📥 Veri Yükleme
Ön işleme adımında üretilen temizlenmiş ama henüz ML modeline girmemiş olan zaman serisi formundaki ham segment verilerini yükleyeceğiz.
*(Not: Eger 02'de doğrudan dataset.csv kullanıldıysa, o verinin üzerine özellik eklenebilir veya ham segments.csv üzerinden özellikler üretilebilir. Burada Reaction Wheel'lerin zamansal özelliklerini tam olarak göstermek için, ham formattaki segments verisini kullanacağız)*

In [ ]:
# Orijinal segments verisini kullanacağız ki rolling ve shift (lag) yapabilelim.
df_segments = pd.read_csv('../data/raw/segments.csv')
df_segments['timestamp'] = pd.to_datetime(df_segments['timestamp'])
df_segments = df_segments.sort_values(by=['channel', 'timestamp']).reset_index(drop=True)

print(f'📊 Yüklenen veri boyutu: {df_segments.shape}')
display(df_segments.head(3))

---
## 🚀 Sınıfı Başlatma ve Özellik Çıkarımı (Tüm Adımlar)

`src/feature_engineer.py` içerisindeki `ReactionWheelFeatureEngineer` sınıfı; 
1. Zaman Alanı
2. Frekans Alanı
3. Fiziksel
4. Multivariate
5. Lag (Gecikmeli)
6. Özellik Seçimi (Varyans ve Korelasyon temelli)

adımlarının tamamını `transform()` metodu ile tek seferde yapacak şekilde tasarlanmıştır.

In [ ]:
# Modelin çalışacağı ana kanallar (Telemetri Sensörleri)
telemetry_channels = ['value'] # Segment dosyasındaki ölçüm sütunu

# Engineer objesini başlat
# 30, 60 ve 120 saniyelik pencereler
# 1, 5, 10, 30 ve 60 adımlık gecikmeler
engineer = ReactionWheelFeatureEngineer(
    rolling_windows=[30, 60, 120],
    lags=[1, 5, 10, 30, 60],
    n_pca_components=3,
    corr_threshold=0.95
)

# Her kanal (channel) verisi bağımsız zaman serisi olduğu için groupby ile ayrı ayrı özellik çıkarıp birleştirelim.
# Hızlı çalışması için şimdilik sadece CADC0872 (Manyetometre) kanalından bir örnek alıyoruz.
# Gerçek senaryoda tüm dataset apply ile işlenmelidir.

sample_df = df_segments[df_segments['channel'] == 'CADC0872'].copy()
print(f"İşlenecek kanal verisi: {sample_df.shape}")

# Tüm özellikleri çıkar ve gereksizleri ele (fit=True)
df_features = engineer.transform(sample_df, columns=telemetry_channels, target_col='anomaly', fit=True)

print('\n=== Üretilen Özelliklerin İlk 5 Satırı ===')
display(df_features.head())

---
## 📊 1. Zaman Alanı (Time Domain) Özellikleri İnceleme

Üretilen özellikleri kategorik olarak görselleştirerek inceleyelim.

In [ ]:
time_cols = [c for c in df_features.columns if 'roll_mean' in c or 'roll_std' in c or 'rms' in c]

fig = go.Figure()
if 'value' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value'], mode='lines', name='Ham Sinyal', line=dict(color='gray', width=1)))
if 'value_roll_mean_60' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value_roll_mean_60'], mode='lines', name='Rolling Mean (w=60)', line=dict(color='red')))
elif 'value_roll_mean_30' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value_roll_mean_30'], mode='lines', name='Rolling Mean (w=30)', line=dict(color='red')))
if 'value_rms_60' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value_rms_60'], mode='lines', name='RMS (w=60)', line=dict(color='blue')))
elif 'value_rms_30' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value_rms_30'], mode='lines', name='RMS (w=30)', line=dict(color='blue')))
fig.update_layout(title='Zaman Alanı Özellikleri (Mean ve RMS)', template='plotly_dark', height=500)
fig.show()



---
## 🔗 2. Değişim (Rate of Change & Jerk)

Türev ve İkinci Türev (Jerk), Reaction Wheel sistemlerinde motorun ani devir (RPM) veya ivmelenme tepkilerini (vibration/spike) ölçmek için kritik özelliklerdir.

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=['Rate of Change (1st Derivative)', 'Jerk (2nd Derivative)'])

if 'value_roc' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value_roc'], mode='lines', name='ROC', line=dict(color='orange')), row=1, col=1)
if 'value_jerk' in df_features.columns:
    fig.add_trace(go.Scatter(y=df_features['value_jerk'], mode='lines', name='Jerk', line=dict(color='purple')), row=2, col=1)

fig.update_layout(title='Reaction Wheel Ani Değişim Metrikleri', template='plotly_dark', height=600)
fig.show()



---
## ⏳ 3. Gecikmeli Özellikler (Lag Features)
Zaman serisinin otoregresif (kendi geçmişine bağlılık) yapısını modellemek için çıkarılan gecikmeli özellikler.

In [ ]:
lag_cols = [c for c in df_features.columns if 'lag' in c]
print("Üretilen Gecikmeli Özellikler:", lag_cols)

lags = [1, 5, 10, 30, 60]
valid_lags = [l for l in lags if f'value_lag_{l}' in df_features.columns and 'value' in df_features.columns]

if valid_lags:
    autocorrs = [df_features['value'].corr(df_features[f'value_lag_{l}']) for l in valid_lags]
    plt.figure(figsize=(10, 5))
    sns.barplot(x=[f'Lag {l}' for l in valid_lags], y=autocorrs, palette='viridis')
    plt.title('Özelliklerin Otokorelasyon Katsayıları (ACF Benzeri)', fontsize=14)
    plt.ylabel('Pearson Korelasyon')
    plt.show()
else:
    print('Lags grafiği çizilemedi çünkü orijinal sinyal veya gecikme özellikleri yüksek korelasyon nedeniyle filtrelenmiş.')



---
## 📉 4. Özellik Seçimi (Feature Selection) Analizi
`engineer.transform` metodu çalışırken Varyans Seçimi (Variance Threshold) ve Korelasyon Seçimi (Pearson > 0.95) uyguladı.

In [ ]:
print("=== Feature Selection Sonuçları ===")
print(f"Üretilen Nihai Özellik Sayısı: {engineer.feature_metadata['total_features_generated']}")
print(f"Yüksek Korelasyon Nedeniyle Düşürülen Özellik Sayısı: {len(engineer.feature_metadata.get('dropped_correlated', []))}")
print("\nDüşürülen Özellikler (Örnek 10 adet):")
print(engineer.feature_metadata.get('dropped_correlated', [])[:10])

print("\n💡 Neden Düşürüldü? Machine Learning modelleri (özellikle lineer olanlar ve sinir ağları) birbirini %95'ten fazla kopyalayan özelliklerden olumsuz etkilenir (Multicollinearity).")

---
## 🎯 5. Özellik Doğrulama (t-SNE / PCA 2D Ayrılabilirlik)

Ürettiğimiz bu zengin özellik matrisi ile Anomali (1) ve Normal (0) sınıfları veri uzayında nasıl ayrılıyor görselleştirelim.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Sadece sayısal özellikleri seç
X_feat = df_features.select_dtypes(include=[np.number]).drop(columns=['segment', 'train', 'anomaly'], errors='ignore')
y_label = df_features['anomaly']

# Ölçeklendirme
X_scaled = StandardScaler().fit_transform(X_feat.fillna(0))

# Çok veri olduğu için hızlandırmak adına örneklem (sample) alıyoruz
sample_idx = np.random.choice(X_scaled.shape[0], min(2000, X_scaled.shape[0]), replace=False)
X_sample = X_scaled[sample_idx]
y_sample = y_label.iloc[sample_idx]

# t-SNE İndirgeme (2 Boyut)
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_sample)

# Görselleştirme
tsne_df = pd.DataFrame({'D1': X_tsne[:, 0], 'D2': X_tsne[:, 1], 'Anomaly': y_sample})

fig = px.scatter(tsne_df, x='D1', y='D2', color='Anomaly', 
                 title='t-SNE ile Özellik Uzayının Ayrılabilirliği (Normal=0, Anomaly=1)',
                 color_continuous_scale=['#2ecc71', '#e74c3c'], opacity=0.7)
fig.update_layout(template='plotly_dark', height=600)
fig.show()

print('💡 Yorum: Mavi (veya Yeşil) bölgeler ile Kırmızı bölgelerin (Anomaliler) kümelenme eğiliminde olması, ürettiğimiz özelliklerin modeli iyi eğitebileceğinin göstergesidir.')

---
## 💾 6. Özellik Matrisini ve Kataloğu Kaydetme

In [ ]:
import json

# Tüm kanalların features matrisini `data/features` altına kaydediyoruz.
# Parquet formatı ile 
df_features.to_parquet('../data/features/reaction_wheel_features.parquet')

# Metadata / Katalog kaydetme
catalog = {
    "feature_count": len(engineer.selected_features),
    "features_list": engineer.selected_features,
    "dropped_correlated": engineer.feature_metadata.get('dropped_correlated', []),
    "parameters_used": {
        "rolling_windows": engineer.rolling_windows,
        "lags": engineer.lags,
        "pca_components": engineer.n_pca_components
    }
}

with open('../data/features/feature_catalog.json', 'w', encoding='utf-8') as f:
    json.dump(catalog, f, indent=4, ensure_ascii=False)

print('✅ Özellik Matrisi ve Katalog başarıyla `data/features/` klasörüne kaydedildi.')

### 6.1 HTML Rapor Export

In [ ]:
# HTML Rapor Olusturma
!jupyter nbconvert --to html 03_feature_engineering.ipynb --output ../reports/03_feature_engineering_rapor.html
print("HTML Raporu reports/03_feature_engineering_rapor.html konumuna kaydedildi.")